In [1]:
!pip install kagglehub

In [2]:
#copy raw SAP files and load target tables into pandas DataFrames
import os
import re
import glob
import json
import shutil
import urllib
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, text, inspect
import kagglehub

In [3]:
#project folders
raw_folder = "SAP_Project/data/raw"
os.makedirs(raw_folder, exist_ok=True)
#target SAP tables
target_tables = ["vbrk", "vbrp", "kna1", "ska1", "skat","cepc", "bkpf", "t001", "cepct", "bseg"]
# business keys per table for duplicate detection
business_keys = {
    "bkpf":  ["bukrs", "belnr", "gjahr"],
    "bseg":  ["bukrs", "belnr", "gjahr", "buzei"],
    "kna1":  ["kunnr"],
    "ska1":  ["ktopl", "saknr"],
    "skat":  ["ktopl", "saknr"],
    "cepc":  ["prctr", "kokrs"],
    "cepct": ["prctr", "kokrs", "spras", "datbi"],
    "vbrk":  ["vbeln"],
    "vbrp":  ["vbeln", "posnr"],
    "t001":  ["bukrs"],
}

# One timestamp for the whole run
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Run timestamp: {RUN_TIMESTAMP}")

Run timestamp: 2026-04-09 01:24:22


In [4]:
#import dataset from kaggle
kaggle_path = kagglehub.dataset_download("mustafakeser4/sap-dataset-bigquery-dataset")

print("Path to dataset files:", kaggle_path)

Path to dataset files: C:\Users\ACER\.cache\kagglehub\datasets\mustafakeser4\sap-dataset-bigquery-dataset\versions\1


In [5]:
# ── Copy CSV files to raw folder ─────────────────────────────
copy_summary = []
for table in target_tables:
    all_files = sorted(glob.glob(os.path.join(kaggle_path, f"{table}*.csv")))
    matched   = [f for f in all_files
                 if re.match(rf"^{table}(_\d+)?\.csv$",
                             os.path.basename(f).lower())]
    if not matched:
        print(f"[WARNING] No file found for: {table}")
        continue
    copied = skipped = 0
    for fp in matched:
        dst = os.path.join(raw_folder, os.path.basename(fp))
        if not os.path.exists(dst):
            shutil.copy2(fp, dst)
            copied += 1
        else:
            skipped += 1
            print(f"[INFO] Already exists, skipped: {os.path.basename(fp)}")
    copy_summary.append({"table": table, "matched": len(matched),
                         "copied": copied, "skipped": skipped})

pd.DataFrame(copy_summary)

[INFO] Already exists, skipped: vbrk.csv
[INFO] Already exists, skipped: vbrp.csv
[INFO] Already exists, skipped: kna1.csv
[INFO] Already exists, skipped: ska1.csv
[INFO] Already exists, skipped: skat.csv
[INFO] Already exists, skipped: cepc.csv
[INFO] Already exists, skipped: bkpf.csv
[INFO] Already exists, skipped: t001.csv
[INFO] Already exists, skipped: cepct.csv
[INFO] Already exists, skipped: bseg.csv


,table,matched,copied,skipped
0,vbrk,1,0,1
1,vbrp,1,0,1
2,kna1,1,0,1
3,ska1,1,0,1
4,skat,1,0,1
5,cepc,1,0,1
6,bkpf,1,0,1
7,t001,1,0,1
8,cepct,1,0,1
9,bseg,1,0,1


In [6]:
# clean data
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase + strip column names."""
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    return df


def trim_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Strip leading/trailing whitespace from all string columns."""
    str_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df[col] = df[col].str.strip()
    return df


def read_csv_with_encoding(file_path: str) -> pd.DataFrame:
    """Try multiple encodings and delimiters; fall back gracefully."""
    encodings  = ["utf-8", "cp874", "tis-620", "latin-1"]
    separators = [None, ";", "\t", "|"]
    for enc in encodings:
        for sep in separators:
            try:
                kwargs = {"dtype": str, "encoding": enc, "sep": sep}
                if sep is None:
                    kwargs["engine"] = "python"
                else:
                    kwargs["low_memory"] = False
                df = pd.read_csv(file_path, **kwargs)
                print(f"[OK] {os.path.basename(file_path)} — enc={enc} sep={repr(sep)}")
                return df
            except (UnicodeDecodeError, LookupError, pd.errors.ParserError):
                continue
    print(f"[WARN] All attempts failed for {file_path}, using fallback")
    return pd.read_csv(file_path, dtype=str, low_memory=False,
                       encoding="latin-1", sep=";", on_bad_lines="skip",
                       engine="python")

def match_sql_columns(df: pd.DataFrame, table_name: str,
                      schema: str, engine) -> pd.DataFrame:
    """Keep only columns that exist in the SQL table DDL. Log dropped columns."""
    inspector   = inspect(engine)
    sql_cols    = {col["name"] for col in inspector.get_columns(table_name, schema=schema)}
    kept        = [c for c in df.columns if c in sql_cols]
    dropped     = [c for c in df.columns if c not in sql_cols]
    if dropped:
        print(f"[INFO] {schema}.{table_name} — dropping {len(dropped)} columns "
              f"not in DDL: {dropped[:10]}{'...' if len(dropped) > 10 else ''}")
    return df[kept]


# Logging
def log_pipeline_start(engine, pipeline_name: str) -> int:
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO stg.pipeline_run_log
                (pipeline_name, run_timestamp, status, note)
            VALUES
                (:name, :ts, 'STARTED', 'Pipeline started')
        """), {"name": pipeline_name, "ts": datetime.now()})
        run_id = conn.execute(text(
            "SELECT MAX(run_id) FROM stg.pipeline_run_log "
            "WHERE pipeline_name = :name"
        ), {"name": pipeline_name}).scalar()
    return run_id


# update end_timestamp and rows_affected on completion
def log_pipeline_end(engine, run_id: int, status: str,
                     rows_affected: int = None, note: str = None):
    with engine.begin() as conn:
        conn.execute(text("""
            UPDATE stg.pipeline_run_log
            SET status        = :status,
                end_timestamp = :end_ts,
                rows_affected = :rows,
                note          = :note
            WHERE run_id = :run_id
        """), {
            "status":  status,
            "end_ts":  datetime.now(),
            "rows":    rows_affected,
            "note":    str(note)[:2000] if note else None,
            "run_id":  run_id,
        })


# check_category and sample_values parameters
def log_validation(engine, run_id: int, table_name: str,
                   check_name: str, check_category: str,
                   issue_count: int, status: str,
                   note: str = None, sample_values=None):
    sample_str = None
    if sample_values:
        try:
            sample_str = json.dumps(sample_values[:5], default=str)[:2000]
        except Exception:
            sample_str = str(sample_values)[:2000]
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO stg.data_validation_log
                (run_id, check_timestamp, table_name, check_name,
                 check_category, issue_count, status, note, sample_values)
            VALUES
                (:run_id, :ts, :tbl, :chk, :cat, :cnt, :status, :note, :sample)
        """), {
            "run_id":  run_id,
            "ts":      datetime.now(),
            "tbl":     table_name,
            "chk":     check_name,
            "cat":     check_category,
            "cnt":     issue_count,
            "status":  status,
            "note":    str(note)[:2000] if note else None,
            "sample":  sample_str,
        })

print("Helpers loaded.")

Helpers loaded.


In [7]:
# ── Load CSVs into memory ─────────────────────────────────────
loaded_tables = {}
load_summary  = []

for table in target_tables:
    all_files     = sorted(glob.glob(os.path.join(raw_folder, "**", "*.csv"), recursive=True))
    matched_files = [
        f for f in all_files
        if re.match(rf"^{re.escape(table)}(_\d+)?\.csv$",
                    os.path.basename(f), flags=re.IGNORECASE)
    ]
    if not matched_files:
        print(f"[WARNING] No file found in raw folder for: {table}")
        continue

    df_list = []
    for fp in matched_files:
        fname  = os.path.basename(fp)
        tmp_df = read_csv_with_encoding(fp)
        tmp_df = clean_columns(tmp_df)
        tmp_df = trim_text_columns(tmp_df)
        tmp_df = tmp_df.assign(
            source_file_name=fname,
            load_timestamp=RUN_TIMESTAMP
        ).copy()

        df_list.append(tmp_df)

    rows_before = sum(len(d) for d in df_list)
    combined    = pd.concat(df_list, ignore_index=True)

    # deduplicate using business keys, not all columns
    bkeys = business_keys.get(table)
    available_bkeys = [k for k in (bkeys or []) if k in combined.columns]
    if available_bkeys:
        # Keep last occurrence (latest source file)
        combined = combined.sort_values("load_timestamp", ascending=False)
        dup_count = combined.duplicated(subset=available_bkeys).sum()
        combined  = combined.drop_duplicates(subset=available_bkeys, keep="first")
    else:
        dup_count = combined.duplicated().sum()
        combined  = combined.drop_duplicates()

    loaded_tables[table] = combined
    load_summary.append({
        "table":            table,
        "file_count":       len(matched_files),
        "rows_before_dedup":rows_before,
        "rows_after_dedup": len(combined),
        "duplicates_dropped": int(dup_count),
        "columns":          combined.shape[1],
        "null_cells":       int(combined.isnull().sum().sum()),
    })

pd.DataFrame(load_summary)

[OK] vbrk.csv — enc=utf-8 sep=None
[OK] vbrp.csv — enc=utf-8 sep=None
[OK] kna1.csv — enc=utf-8 sep=None
[OK] ska1.csv — enc=utf-8 sep=None
[OK] skat.csv — enc=utf-8 sep=None
[OK] cepc.csv — enc=utf-8 sep=None
[OK] bkpf.csv — enc=utf-8 sep=None
[OK] t001.csv — enc=utf-8 sep=None
[OK] cepct.csv — enc=utf-8 sep=None
[OK] bseg.csv — enc=utf-8 sep=None


,table,file_count,rows_before_dedup,rows_after_dedup,duplicates_dropped,columns,null_cells
0,vbrk,1,89692,40323,49369,120,2816110
1,vbrp,1,119282,59149,60133,229,8942936
2,kna1,1,20209,19942,267,194,3090288
3,ska1,1,146692,29046,117646,23,283723
4,skat,1,782051,33313,748738,12,12793
5,cepc,1,1451,849,602,53,31727
6,bkpf,1,150057,136891,13166,127,13019749
7,t001,1,605,77,528,85,4753
8,cepct,1,5989,5369,620,13,1109
9,bseg,1,332106,291234,40872,358,92289224


In [8]:
!pip install sqlalchemy pyodbc

In [9]:
for name, df in loaded_tables.items():
    print(name, df.shape, df.columns.duplicated().sum())

vbrk (40323, 120) 0
vbrp (59149, 229) 0
kna1 (19942, 194) 0
ska1 (29046, 23) 0
skat (33313, 12) 0
cepc (849, 53) 0
bkpf (136891, 127) 0
t001 (77, 85) 0
cepct (5369, 13) 0
bseg (291234, 358) 0


In [10]:
loaded_tables["bseg"].head()

,mandt,bukrs,belnr,gjahr,buzei,buzid,augdt,augcp,augbl,bschl,...,inward_dt,ground_no,ground_dt,ground_typ,pymtkey,operation_flag,is_deleted,recordstamp,source_file_name,load_timestamp
0,100,C003,5105600151,2022,002,S,NaN,NaN,NaN,81,...,NaN,NaN,NaN,NaN,NaN,I,False,2022-04-26 14:35:07.788536+00:00,bseg.csv,2026-04-09 01:24:22
221391,250,EU01,5100022364,2020,002,W,NaN,NaN,NaN,86,...,NaN,NaN,NaN,NaN,NaN,I,False,2022-03-26 02:45:07.634110+00:00,bseg.csv,2026-04-09 01:24:22
221409,250,EU01,5100022412,2020,002,W,NaN,NaN,NaN,86,...,NaN,NaN,NaN,NaN,NaN,I,False,2022-03-26 02:45:07.634110+00:00,bseg.csv,2026-04-09 01:24:22
221408,250,EU01,5100022411,2020,002,W,NaN,NaN,NaN,86,...,NaN,NaN,NaN,NaN,NaN,I,False,2022-03-26 02:45:07.634110+00:00,bseg.csv,2026-04-09 01:24:22
221407,250,EU01,5100022407,2020,002,W,NaN,NaN,NaN,86,...,NaN,NaN,NaN,NaN,NaN,I,False,2022-03-26 02:45:07.634110+00:00,bseg.csv,2026-04-09 01:24:22


In [11]:
# ── Connect to SQL Server ─────────────────────────────────────
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=financial_sap_project;"
    "Trusted_Connection=yes;"
)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True,
)
with engine.connect() as conn:
    db = conn.execute(text("SELECT DB_NAME()")).scalar()
    print(f"Connected: {db}")

Connected: financial_sap_project


In [12]:
# ingestion pipeline ───────────────────────────────────
table_mapping = {
    "bkpf":  "stg_bkpf",
    "bseg":  "stg_bseg",
    "kna1":  "stg_kna1",
    "ska1":  "stg_ska1",
    "skat":  "stg_skat",
    "cepc":  "stg_cepc",
    "cepct": "stg_cepct",
    "vbrk":  "stg_vbrk",
    "t001":  "stg_t001",
    "vbrp":  "stg_vbrp",
}

PIPELINE_NAME = "stg_load_pipeline"
total_rows_loaded = 0

run_id = log_pipeline_start(engine, PIPELINE_NAME)
print(f"Pipeline started — run_id={run_id}")

try:
    for py_table, sql_table in table_mapping.items():
        if py_table not in loaded_tables:
            print(f"[SKIP] {py_table} not in loaded_tables")
            continue

        df = loaded_tables[py_table].copy()

        # Column alignment ──────────────────────────────────
        df = match_sql_columns(df, sql_table, "stg", engine)

        # per-table business-key duplicate check (logs to validation log)
        bkeys      = business_keys.get(py_table, [])
        avail_keys = [k for k in bkeys if k in df.columns]
        if avail_keys:
            dup_mask   = df.duplicated(subset=avail_keys, keep="first")
            dup_count  = int(dup_mask.sum())
            sample_dup = df[dup_mask][avail_keys].head(5).to_dict("records") if dup_count else []
            log_validation(
                engine, run_id,
                table_name      = f"stg.{sql_table}",
                check_name      = "business_key_duplicate_check",
                check_category  = "DUPLICATE",
                issue_count     = dup_count,
                status          = "WARNING" if dup_count > 0 else "PASSED",
                note            = f"{dup_count} duplicate business keys found (keys: {avail_keys})",
                sample_values   = sample_dup,
            )

        # NULL check on mandatory key columns
        for key_col in avail_keys[:2]:   # check first 2 business key columns
            null_mask  = df[key_col].isna() | (df[key_col].str.strip() == "")
            null_count = int(null_mask.sum())
            if null_count > 0:
                log_validation(
                    engine, run_id,
                    table_name     = f"stg.{sql_table}",
                    check_name     = f"null_key_check_{key_col}",
                    check_category = "NULL_KEY",
                    issue_count    = null_count,
                    status         = "WARNING",
                    note           = f"{null_count} rows with NULL/empty '{key_col}'",
                )

        # Truncate existing staging data ────────────────────
        with engine.begin() as conn:
            conn.execute(text(f"DELETE FROM stg.{sql_table}"))

        # to_sql() with fast_executemany — parameterized, fast, no SQL injection
        df.to_sql(
            name        = sql_table,
            con         = engine,
            schema      = "stg",
            if_exists   = "append",   # table already created by DDL script
            index       = False,
            chunksize   = 1000
        )

        # verify row count in DB matches DataFrame
        with engine.connect() as conn:
            db_count = conn.execute(
                text(f"SELECT COUNT(*) FROM stg.{sql_table}")
            ).scalar()

        count_match = db_count == len(df)
        log_validation(
            engine, run_id,
            table_name     = f"stg.{sql_table}",
            check_name     = "row_count_verification",
            check_category = "ROW_COUNT",
            issue_count    = 0 if count_match else abs(db_count - len(df)),
            status         = "PASSED" if count_match else "FAILED",
            note           = f"DataFrame={len(df):,} rows | DB={db_count:,} rows",
        )

        if not count_match:
            raise RuntimeError(
                f"Row count mismatch for stg.{sql_table}: "
                f"expected {len(df)}, got {db_count}"
            )

        total_rows_loaded += db_count
        print(f"[OK] {py_table} → stg.{sql_table} | {db_count:,} rows")

    # update pipeline log with end time and total rows
    log_pipeline_end(
        engine, run_id,
        status        = "SUCCESS",
        rows_affected = total_rows_loaded,
        note          = f"All {len(table_mapping)} tables loaded. Total rows: {total_rows_loaded:,}",
    )
    print(f"\nPipeline SUCCESS — run_id={run_id} | total rows={total_rows_loaded:,}")

except Exception as exc:
    log_pipeline_end(engine, run_id, status="FAILED", note=str(exc))
    print(f"Pipeline FAILED: {exc}")
    raise

Pipeline started — run_id=1
[INFO] stg.stg_bkpf — dropping 115 columns not in DDL: ['monat', 'cpudt', 'cputm', 'aedat', 'upddt', 'wwert', 'tcode', 'bvorg', 'dbblg', 'stblg']...
[OK] bkpf → stg.stg_bkpf | 136,891 rows
[INFO] stg.stg_bseg — dropping 331 columns not in DDL: ['buzid', 'augcp', 'umsks', 'zumsk', 'gsber', 'pargb', 'mwskz', 'qsskz', 'kzbtr', 'pswbt']...
[OK] bseg → stg.stg_bseg | 291,234 rows
[INFO] stg.stg_kna1 — dropping 185 columns not in DDL: ['mandt', 'name2', 'pstlz', 'sortl', 'stras', 'telf1', 'telfx', 'xcpdk', 'adrnr', 'mcod1']...
[OK] kna1 → stg.stg_kna1 | 19,942 rows
[INFO] stg.stg_ska1 — dropping 16 columns not in DDL: ['mandt', 'sakan', 'erdat', 'ernam', 'gvtyp', 'mustr', 'vbund', 'xloev', 'xspea', 'xspeb']...
[OK] ska1 → stg.stg_ska1 | 29,046 rows
[INFO] stg.stg_skat — dropping 6 columns not in DDL: ['mandt', 'spras', 'mcod1', 'operation_flag', 'is_deleted', 'recordstamp']
[OK] skat → stg.stg_skat | 33,313 rows
[INFO] stg.stg_cepc — dropping 38 columns not in DDL

In [13]:
# Post-load summary ─────────────────────────────────────────
with engine.connect() as conn:
    run_summary = pd.read_sql(
        text("""
            SELECT
                r.run_id,
                r.pipeline_name,
                r.run_timestamp,
                r.end_timestamp,
                r.duration_seconds,
                r.status,
                r.rows_affected,
                SUM(CASE WHEN v.status = 'FAILED'  THEN 1 ELSE 0 END) AS failed_checks,
                SUM(CASE WHEN v.status = 'WARNING' THEN 1 ELSE 0 END) AS warning_checks,
                SUM(CASE WHEN v.status = 'PASSED'  THEN 1 ELSE 0 END) AS passed_checks
            FROM stg.pipeline_run_log r
            LEFT JOIN stg.data_validation_log v ON r.run_id = v.run_id
            WHERE r.run_id = :run_id
            GROUP BY r.run_id, r.pipeline_name, r.run_timestamp,
                     r.end_timestamp, r.duration_seconds, r.status, r.rows_affected
        """),
        conn,
        params={"run_id": run_id},
    )

print(run_summary.to_string(index=False))

 run_id     pipeline_name           run_timestamp           end_timestamp  duration_seconds  status  rows_affected  failed_checks  warning_checks  passed_checks
      1 stg_load_pipeline 2026-04-09 01:26:39.993 2026-04-09 01:27:30.740                51 SUCCESS         616193              0               0             20


In [14]:
# Show any FAILED or WARNING checks
with engine.connect() as conn:
    issues = pd.read_sql(
        text("""
            SELECT table_name, check_category, check_name,
                   issue_count, status, note, sample_values
            FROM stg.data_validation_log
            WHERE run_id = :run_id
              AND status IN ('FAILED', 'WARNING')
            ORDER BY status DESC, issue_count DESC
        """),
        conn,
        params={"run_id": run_id},
    )

if issues.empty:
    print("No issues found — all checks PASSED.")
else:
    print(f"{len(issues)} issue(s) found:")
    display(issues)

No issues found — all checks PASSED.
